# Wireless Networking and Mobile Computing

### 1. Introduction

Li-Fi (Light Fidelity) is a wireless technology that uses visible, infrared, or ultraviolet light instead of radio waves to transmit data. Based on IEEE 802.11 and 802.15.7 standards, it works by rapidly modulating an LED’s brightness so fast that the eye cannot detect it, while a photodiode or camera sensor receives and decodes the signal. Li-Fi can reach high data rates and offers security advantages since light does not pass through walls, but it requires line-of-sight, fails in darkness, and provides limited mobility. It is useful in areas like hospitals, aircraft, and factories where radio use is restricted.

Li-Fi is part of Visible Light Communication (VLC), which sends data through fast light modulation. A simple form is LED-to-LED communication, where one LED transmits and another detects light when reverse-biased. This setup is low-cost and energy-efficient, suitable for short-range or embedded systems such as toys and sensors, though it has shorter range and lower sensitivity than VLC systems using photodiodes.

<center>
  <img src="../../Admin/picture/vlc_toys.png" 
       alt="VLC Toys" 
       width="40%"
       style="filter: grayscale(80%);">
  <p><em>Interactive toys with low cost LED-to-LED communication.</em></p>
</center>

This notebook allows tests with one or two prototype boards.

#### 1.1 Literature

[<a id="ref01">01</a>] P. Dietz, W. Yerazunis, and D. Leigh, **Very Low-Cost Sensing and Communication Using Bidirectional LEDs**, Technical Report, MERL, 2003.  

[<a id="ref02">02</a>] S. Schmid, G. Corbellini, S. Mangold, and T. R. Gross, **Continuous Synchronization for LED-to-LED Visible Light Communication Networks**, 3rd International Workshop on Optical Wireless Communications (IWOW), September 2014, Madeira, Portugal.  

[<a id="ref03">03</a>] S. Schmid, G. Corbellini, S. Mangold, and T. R. Gross, **LED-to-LED Visible Light Communication Networks**, ACM MobiHoc, 29 July – 1 August 2013, Bangalore, India.  

[<a id="ref04">04</a>] Background Reading "[VLC](http://gitlab.ethz.ch/wireless/BackgroundReading/-/tree/main/VLC,%20Visible%20Light%20Communication)"  


### 2. Preparing the microcontroller boards
#### 2.1 Background Information

The Arduino Uno R3 is a widely used microcontroller board based on the ATmega328P. It provides 14 digital I/O pins, six analog inputs, a 16 MHz clock, and USB/power connections. The board runs on USB or external power and is typically programmed via the Arduino IDE, though for our experiments it is pre-flashed and does not require programming. Its simplicity, low cost, and large community make it ideal for rapid prototyping and education.


<center>
<img src="../../Admin/picture/arduino_uno_r3.png" 
     alt="Arduino" 
     width="40%" 
     style="filter: grayscale(40%);">
<img src="../../Admin/picture/LED.png" 
     alt="Arduino" 
     width="15%" 
     style="filter: grayscale(40%);">
     <p><em>Arduino Uno R3 and one LED. Note the A0 and A1 connectors, which is where the LED must be connected to.</em></p>
</center>


#### 2.2 Handling the Arduino safely

An Arduino runs at low voltages and can safely be touched. You will not get an electric shock from the board itself. The real risks are short circuits, hot parts, and external power/equipment. 

- Always place the board on a **non-conductive base** (acrylic, cardboard). Don’t lay it on metal, do not use conductive material (scissors) to work on it. 
- Avoid touching pin headers, USB connector shells, and components unnecessarily. 
- **Power off to rewire:** Unplug USB / power **before** changing jumpers, shields, or sensors. 
- **Avoid shorts:** Keep loose wires trimmed; don’t drop screws/paperclips onto the board. 
- **Use the laptop’s USB for power**; it limits current. 
- **Keep liquids away!** 
- A wrong connection rarely destroys everything. 
- If something misbehaves: disconnect and power off → inspect → fix → re-connect.

#### 2.3 Setup instructions

For each VLC device, take one red LED and connect its **Cathode** to pin **A0** and its **Anode** to pin **A1**. Connect the Arduino to your computer using USB. When powered, the LED will light up, indicating that the Arduino is receiving power.

#### 2.4 Serial interface

##### 2.4.1 Python Setup Test

In [ ]:
# --- Install ---
%pip install -U pyserial

In [ ]:
# --- Diagnostics ---
import sys, os, glob
import serial
print("Imported from:", getattr(serial, "__file__", serial))
print("Has Serial class:", hasattr(serial, "Serial"))
print("Local files that could shadow it:", glob.glob("serial*"))

If "Has Serial class" is False or the path is not `site-packages/pyserial`, then run the next cell to check further:

In [ ]:
# --- Diagnostic ---
import importlib, sys, os
import serial as _serial
if not hasattr(_serial, "Serial"):
    msg = (f"Wrong 'serial' module loaded from: {getattr(_serial,'__file__',_serial)}\n"
           "Fix: %pip uninstall -y serial ; %pip uninstall -y pyserial ; %pip install -U pyserial ; then restart kernel.")
    raise ImportError(msg)
else:
    print("PySerial version:", _serial.__version__, " - We are good to go! :-)")
serial = _serial  # use the verified module

##### 2.4.2 Serial port autodetection
The next code cell tries to autodetect the available serial port(s). It also updates the port names. The USB-to-serial interface enables sending commands and data to the VLC device and read responses back. On macOS, the board is usually detected automatically as a serial port. On Windows, it might be required to install the Arduino USB driver first (see [the guide](http://arduino.cc/en/Guide/Windows#toc4)). After installation, unplug/replug the board and re-run the auto-detection.

In case the auto-detection failed, we can manually set the port names (e.g., `PORT_A = 'COM4'`, `PORT_B = 'COM5'` on Windows, or the corresponding `/dev/tty…` paths on macOS). 

- On  Windows, it looks like `COM##` and can be found in the Device Manager - Ports (COM & LPT). 
- On MacOS, it looks like `/dev/tty.usbmodem*` or `/dev/tty.usbserial*` and can be identified through `ls /dev/tty.*`.

In [ ]:
# --- Auto-detect for serial ports (Windows / macOS / Linux) ---
from serial.tools import list_ports
import platform, re

KNOWN_VIDS = {
    0x2341, 0x2A03,  # Arduino
    0x1A86,          # WCH CH340
    0x10C4,          # Silicon Labs CP210x
    0x0403,          # FTDI
    0x239A,          # Adafruit
    0x16C0,          # PJRC / Teensy
    0x2E8A,          # Raspberry Pi (RP2040)
}

# macOS exposes fake ports we should ignore
SKIP_SUBSTR = {"bluetooth", "incoming", "debug-console", "debug"}

def _com_number(dev):
    m = re.match(r"^COM(\d+)$", (dev or ""), re.IGNORECASE)
    return int(m.group(1)) if m else -1

def _is_candidate(p):
    dev  = (p.device or "").lower()
    desc = (p.description or "").lower()
    manu = (p.manufacturer or "").lower()
    vid  = getattr(p, "vid", None)

    if any(s in dev for s in SKIP_SUBSTR) or any(s in desc for s in SKIP_SUBSTR):
        return False

    if vid in KNOWN_VIDS:
        return True

    # common hints even without a known VID
    hints = ("arduino","usbserial","usb modem","usbmodem","ch340","cp210","ftdi","teensy","sparkfun","seeed")
    if any(h in desc or h in manu for h in hints):
        return True

    # any real USB device with VID/PID
    if (getattr(p, "vid", None) is not None) and (getattr(p, "pid", None) is not None):
        return True

    return False

def _score(p):
    dev  = (p.device or "").lower()
    desc = (p.description or "").lower()
    manu = (p.manufacturer or "").lower()
    vid  = getattr(p, "vid", None)
    s = 0.0

    if vid in KNOWN_VIDS: s += 10
    if "arduino" in desc or "arduino" in manu: s += 6
    if any(k in desc for k in ("usbmodem","usbserial","ch340","cp210","ftdi")): s += 4

    # macOS: prefer call-out ports (/dev/cu.*) and usbmodem
    if platform.system() == "Darwin":
        if "/cu."  in dev: s += 2
        if "usbmodem" in dev: s += 2

    # Windows: slightly prefer higher COM numbers (often most recent)
    s += max(_com_number(p.device), 0) * 0.01
    return s

ports = list(list_ports.comports())

if not ports:
    print("No serial ports found. Plug in your device and re-run this cell.")
else:
    ranked = sorted(ports, key=_score, reverse=True)

    print("Detected serial ports:")
    for i, p in enumerate(ranked, 1):
        vid = f"{getattr(p,'vid',None):04X}" if getattr(p,'vid',None) is not None else "----"
        pid = f"{getattr(p,'pid',None):04X}" if getattr(p,'pid',None) is not None else "----"
        print(f"{i:2d}. {p.device:>24}  {p.description:35.35}  VID:PID={vid}:{pid}  Mfr={p.manufacturer or '-'}")

    # filter to real candidates (skip Bluetooth/debug)
    cands = [p for p in ranked if _is_candidate(p)]

    PORT_A = cands[0].device if len(cands) >= 1 else None
    PORT_B = cands[1].device if len(cands) >= 2 else None

    print("\nSelected:")
    print(f"  PORT_A = {PORT_A!r}")
    print(f"  PORT_B = {PORT_B!r}  # None if no second suitable device found")

### 3. Experiments
 
We will use ETH Zurich’s Visible Light Communication (VLC) Protocol commands throughout the notebook. Before proceeding, please note the three reference tables on that wiki page: Commands, Configuration Commands, Message Events.

#### 3.1 Setup


In [ ]:
# --- Bidirectional serial bridge setup ---
import sys, time, threading, re, platform
import serial

BAUD   = 115200
BRIDGE = False
START_RETRIES  = 1
REOPEN_RETRIES = 1
RETRY_DELAY_S  = 1.0
MAX_LINE_BYTES = 8192

# ---- helpers ----
def _valid_port(p): return isinstance(p, str) and p.strip() != ""

def _darwin_alternates(port):
    """On macOS, try the sibling device (/cu.<x> <-> /tty.<x>)."""
    if platform.system() != "Darwin" or not _valid_port(port):
        return []
    if "/cu." in port:
        return [port.replace("/cu.", "/tty.", 1)]
    if "/tty." in port:
        return [port.replace("/tty.", "/cu.", 1)]
    return []

def _try_open_once(port, baud, timeout=0.05, verbose_label=None):
    """Try to open 'port', then (on macOS) its sibling. Return (ser, used_port)."""
    tried = []
    for candidate in [port] + _darwin_alternates(port):
        if not _valid_port(candidate):
            continue
        try:
            s = serial.Serial(candidate, baud, timeout=timeout)
            time.sleep(0.15)
            try:
                s.reset_input_buffer(); s.reset_output_buffer()
            except Exception:
                pass
            if verbose_label:
                print(f"[{verbose_label}] opened on {candidate}")
            return s, candidate
        except Exception as e:
            tried.append((candidate, str(e)))
    # verbose diagnostics
    if verbose_label and tried:
        why = "; ".join([f"{c}: {msg}" for c, msg in tried])
        print(f"[{verbose_label}] open failed ({why})", file=sys.stderr)
    return None, port

def close_if_open(ser):
    try:
        if ser and getattr(ser, "is_open", False):
            try: ser.dtr = False
            except Exception: pass
            try: ser.flush()
            except Exception: pass
            ser.close()
            time.sleep(0.2)
    except NameError:
        pass

# ---- read ports from earlier autodetect (or user) ----
PORT_A = globals().get("PORT_A")
PORT_B = globals().get("PORT_B")

if not _valid_port(PORT_A):
    print("[WARN] PORT_A is not set. Define it or run the autodetect cell.", file=sys.stderr)
if not _valid_port(PORT_B):
    print("[INFO] Only one device expected; PORT_B not set.", file=sys.stderr)

# ---- close any prior handles ----
close_if_open(globals().get("serA"))
close_if_open(globals().get("serB"))

# ---- initial open with macOS fallback ----
serA, PORT_A = _try_open_once(PORT_A, BAUD, timeout=0.05, verbose_label="PORT_A") if _valid_port(PORT_A) else (None, PORT_A)
serB, PORT_B = _try_open_once(PORT_B, BAUD, timeout=0.05, verbose_label="PORT_B") if _valid_port(PORT_B) else (None, PORT_B)

if _valid_port(PORT_A) and not (serA and serA.is_open):
    print(f"[{PORT_A}] not available; will retry once.", file=sys.stderr)
if _valid_port(PORT_B) and not (serB and serB.is_open):
    print(f"[{PORT_B}] not available; will retry once.", file=sys.stderr)

stop_event = threading.Event()
serA_ref = [serA]; serB_ref = [serB]

def pump(name, src_ref, dst_ref, start_retries=1, reopen_retries=1, retry_delay=1.0):
    """Read lines from src_ref[0]; bridge bytes to dst_ref[0] if BRIDGE."""
    if not _valid_port(name):
        print(f"[{name or 'UNKNOWN'}] invalid port; listener not started.", file=sys.stderr)
        return

    buf = bytearray()
    start_left  = start_retries if (src_ref[0] is None or not getattr(src_ref[0], "is_open", False)) else 0
    reopen_left = reopen_retries

    def open_once(label):
        ser, used = _try_open_once(label, BAUD, timeout=0.05)  # tries sibling on macOS
        if ser:
            src_ref[0] = ser
            # Update 'name' to the actually used device for future logs
            return used
        return None

    # initial ensure-open with bounded retries
    current_label = name
    while not stop_event.is_set():
        s = src_ref[0]
        if s is None or not getattr(s, "is_open", False):
            if start_left > 0:
                time.sleep(retry_delay)
                used = open_once(current_label)
                if used:
                    current_label = used
                    break
                start_left -= 1
                continue
            print(f"[{current_label}] unavailable after {START_RETRIES+1} attempts. Stopping listener.", file=sys.stderr)
            return
        break

    while not stop_event.is_set():
        try:
            b = src_ref[0].read(1)
            if not b:
                continue

            if BRIDGE and dst_ref[0] and getattr(dst_ref[0], "is_open", False):
                try: dst_ref[0].write(b)
                except serial.SerialException: pass

            if b == b'\n':
                line = buf.decode('utf-8', errors='replace').rstrip('\r')

                # >>> Hook: forward complete line to notebook callback if defined
                cb = globals().get("on_serial_line")
                if callable(cb):
                    try:
                        cb(current_label, line)
                    except Exception as e:
                        print(f"[{current_label} hook warn] {e}", file=sys.stderr)
                # <<< end hook

                print(f"[{current_label}] {line}")
                buf.clear()
            else:
                buf.extend(b)
                if len(buf) > MAX_LINE_BYTES:
                    line = buf.decode('utf-8', errors='replace')[:MAX_LINE_BYTES] + "…"
                    print(f"[{current_label} warn] oversized line truncated: {line}", file=sys.stderr)
                    buf.clear()

        except serial.SerialException as e:
            print(f"[{current_label} warn] {e}.", file=sys.stderr)
            close_if_open(src_ref[0]); src_ref[0] = None
            if reopen_left > 0:
                time.sleep(retry_delay)
                used = open_once(current_label)
                if used:
                    current_label = used
                    reopen_left = reopen_retries
                    continue
                reopen_left -= 1
                continue
            print(f"[{current_label}] could not reopen (max {reopen_retries} tries). Stopping listener.", file=sys.stderr)
            return

        except AttributeError as e:
            if 'hEvent' in str(e):  # Windows overlapped I/O glitch
                close_if_open(src_ref[0]); src_ref[0] = None
                if reopen_left > 0:
                    time.sleep(retry_delay)
                    used = open_once(current_label)
                    if used:
                        current_label = used
                        reopen_left -= 1
                        continue
                    reopen_left -= 1
                    continue
                print(f"[{current_label}] hEvent glitch and reopen failed. Stopping listener.", file=sys.stderr)
                return
            print(f"[{current_label} pump error] {e}", file=sys.stderr); return

        except Exception as e:
            msg = str(e)
            if ('byref' in msg) or ('ctypes' in msg) or ('OSError' in msg):
                close_if_open(src_ref[0]); src_ref[0] = None
                if reopen_left > 0:
                    time.sleep(retry_delay)
                    used = open_once(current_label)
                    if used:
                        current_label = used
                        reopen_left -= 1
                        continue
                    reopen_left -= 1
                    continue
                print(f"[{current_label}] handle glitch and reopen failed. Stopping listener.", file=sys.stderr)
                return
            print(f"[{current_label} pump error] {e}", file=sys.stderr); return

# start listeners (only for ports that exist)
threads = []
if _valid_port(PORT_A):
    tA = threading.Thread(target=pump, args=(PORT_A, serA_ref, serB_ref, START_RETRIES, REOPEN_RETRIES, RETRY_DELAY_S), daemon=True)
    tA.start(); threads.append(('A', tA))
if _valid_port(PORT_B):
    tB = threading.Thread(target=pump, args=(PORT_B, serB_ref, serA_ref, START_RETRIES, REOPEN_RETRIES, RETRY_DELAY_S), daemon=True)
    tB.start(); threads.append(('B', tB))

if not threads:
    print("[WARN] No valid ports configured. No listeners started.", file=sys.stderr)
else:
    print("Listening setup done. " + " ".join([f"{n}={t.is_alive()}" for n,t in threads]))


# --- sendA/sendB wrappers (robust; no dependency on _send signature) ---
import sys, time

def _write_serial(sref, label, cmd, *, newline=True, delay=0.1, encoding="utf-8"):
    if newline and not cmd.endswith('\n'):
        cmd += '\n'
    s = sref[0]
    if s and getattr(s, "is_open", False):
        s.write(cmd.encode(encoding, errors='replace'))
        if delay:
            time.sleep(delay)
    else:
        print(f"[{label}] not open; send skipped.", file=sys.stderr)

def sendA(cmd: str, *, newline=True, delay=0.1):
    label = PORT_A if ('PORT_A' in globals() and isinstance(PORT_A, str) and PORT_A.strip()) else "PORT_A"
    _write_serial(serA_ref, label, cmd, newline=newline, delay=delay)

def sendB(cmd: str, *, newline=True, delay=0.1):
    label = PORT_B if ('PORT_B' in globals() and isinstance(PORT_B, str) and PORT_B.strip()) else "PORT_B"
    _write_serial(serB_ref, label, cmd, newline=newline, delay=delay)


#### 3.2 Basic tests

There are three basic tests that we can use to switch our LED(s) off and on. While running the short test scripts below, we will  see a number of commands and message events printed out. Please check again the protocol to understand their meaning format. Message events are of form `m[X(,type,message)]`. Message events `m[T]` and `m[R,…]` are always accompanied by statistic events of the form `s[mode,type,src->dest,size(txsize),seq,cw,cwsize,dispatch,time]`, where `cw`, `cwsize` and `dispatch` are only available for data transmission events (`m[T]`). This is explained in detail in the Wiki.

Now please proceed with the simple tests:

In [ ]:
# --- TEST 1: switch off LED(s) ---
print("TEST 1: Switching off LED(s)")
sendA("c[0,3,0]")
sendB("c[0,3,0]")

In [ ]:
# --- TEST 2: switch LED(s) back on ---
print("TEST 2: Switching on LED(s)")
sendA("c[0,3,1]")
sendB("c[0,3,1]")

In [ ]:
# --- TEST 3: alternate blinking of both LEDs ---
print("TEST 3: Alternate blinking")

import time

# helpers for clarity
def led_on(send_fn):
    send_fn("c[0,3,0]", newline=True, delay=0)  # 0 = ON

def led_off(send_fn):
    send_fn("c[0,3,1]", newline=True, delay=0)  # 1 = OFF

def alternate_blink(cycles=10, period=0.2):
    """
    Alternate LEDs on PORT_A and PORT_B.
    Each half-step lasts `period` seconds:
      - PORT_A ON, PORT_B OFF
      - PORT_A OFF, PORT_B ON
    Repeats `cycles` times (so total duration ≈ cycles * 2 * period).
    """
    # start from a known state
    led_off(sendA)
    led_off(sendB)
    time.sleep(0.1)

    for _ in range(cycles):
        # PORT_A ON, PORT_B OFF
        led_on(sendB)
        led_off(sendA)
        time.sleep(period)

        # PORT_A OFF, PORT_B ON
        led_off(sendB)
        led_on(sendA)
        time.sleep(period)

    # optional: finish with both OFF
    led_off(sendA)
    led_off(sendB)

# run it
alternate_blink(cycles=3, period=0.3)


#### 3.3 VLC Networking (free space optical communication with LEDs)
Below we will try to communicate with light. It is suggested to clear all cell outputs that showed up so far so that we can concentrate on the rest of the notebook. The next cell only opens a new field that will then be used to display all outputs, even from subsequent code cells further down below. This is an artifact of the serial connection.

##### 3.3.1 Log Screen

In [ ]:
# --- LOG SCREEN (VLC Networking) ---

import sys, threading, html
import ipywidgets as widgets
from IPython.display import display
from collections import deque

BG = "#d6dde1"     # background (light blue)
FG = "#3e4f58"     # foreground (text) color
HEIGHT = 400       # px
TAIL_LINES = 400   # show last N lines

# 1) Render the log as HTML with inline styles (works in VS Code)
_STYLE = (
    f"background:{BG}; color:{FG}; height:{HEIGHT}px; overflow-y:auto; "
    "border:0; padding:8px; font-family:ui-monospace,monospace; "
    "font-size:10px; line-height:1.25;"
)
log = widgets.HTML(value=f'<div style="{_STYLE}"><pre style="margin:0; color:inherit;"></pre></div>')
display(log)

# 2) Tail buffer that updates the HTML (no print -> no recursion)
class _TailHTML:
    def __init__(self, widget, max_lines=TAIL_LINES):
        self.w = widget
        self.max_lines = max_lines
        self.lines = deque(maxlen=max_lines)
        self.partial = ""
        self.lock = threading.Lock()

    def _render(self):
        text = "\n".join(list(self.lines) + ([self.partial] if self.partial else []))
        esc = html.escape(text)
        self.w.value = f'<div style="{_STYLE}"><pre style="margin:0; color:inherit;">{esc}</pre></div>'

    def write(self, s):
        if not s: return
        with self.lock:
            if "\r" in s:
                s = s.split("\r")[-1]
                self.partial = ""
            self.partial += s
            while "\n" in self.partial:
                line, self.partial = self.partial.split("\n", 1)
                self.lines.append(line)
            self._render()

    def clear(self):
        with self.lock:
            self.lines.clear()
            self.partial = ""
            self._render()

TAIL = _TailHTML(log, max_lines=TAIL_LINES)

# 3) Hook stdout (optionally stderr) into the HTML log
_ORIG_STDOUT, _ORIG_STDERR = sys.stdout, sys.stderr

class _Sink:
    def __init__(self, tail): self.tail = tail
    def write(self, s): self.tail.write(s)
    def flush(self): pass

sys.stdout = _Sink(TAIL)
# To capture stderr too, uncomment:
# sys.stderr = _Sink(TAIL)

def stop_capture():
    sys.stdout = _ORIG_STDOUT
    sys.stderr = _ORIG_STDERR
    print("Capture stopped.")

def clear_log():
    TAIL.clear()

print( "VLC Networking:")

Now please proceed with configuring the (one or two) connected VLC device(s):

##### 3.3.1 Configuration

The following few lines set the basic configuration. Try to understand the commands. In the configuration, there is a device address defined that should be unique in the network. The format of the ID is in hexadecimal, and valid examples are `CD` or `AB`. In general, there is a 8 bit address room, valid addresses are between `00` and `FF`, with the latter one being the broadcast address.

In [ ]:
print(""); print("Set address of first VLC device:");
sendA("a[AB]")
print("MAC, set number of retry attempts:");
sendA("c[1,0,3]")
print("PHY, set FEC threshold:");
sendA("c[0,1,30]")
print(""); print("Set address of second VLC device:");
sendB("a[CD]")
print("MAC, set number of retry attempts:");
sendB("c[1,0,3]")
print("PHY, set FEC threshold:");
sendB("c[0,1,30]")

##### 3.3.2 Send and receive

 VLC data packets sent by a transmitting device to a receiving device's address are acknowledged by the receiving device. This is referred to as "unicast communication". If no `ACK` is received by the transmitter, after some time, packet transmissions are repeated until the maximum number of retransmissions is reached. Packets sent to the broadcast address (`FF`) are received by all devices and not acknowledged or retransmitted. This is referred to as "broadcast communication". 

 Note the following unicast command. Its destination address should not be the same as the transmitter's address. In the log field above, acknowledge frames should appear when data was successfully transmitted (and acknowledged by the receiver).

**3.3.2.1 One unicast and one broadcast message**

Make sure the intended unicast receiver has the right address. 



In [ ]:
print(""); print("hello world (unicast) ...")
sendA("m[hello world in unicast!\0,CD]")
time.sleep(1)
print(""); print("hello world (broadcast) ...")
sendA("m[hello world in broadcast!\0,FF]")


**3.3.2.2 Repeated broadcast messages**

Continuous broadcast, repeatedly. While this is running through serial A, we can check serial B, or the other device (different laptop). Stop the broadcasting by interrupting the cell execution.

In [ ]:
import time

try:
    while True:
        sendA("m[Hoi!\0,FF]")
        time.sleep(2)
except KeyboardInterrupt:
    print("Broadcast interrupted by user.")


Before continuing, stop the cell execution above. It is also recommended to clear all outputs.

#### 4. Chat over VLC

Use the following cells to chat over VLC. Make sure the destination address and the port are correct. Type your entries in the console (section 4.2). As before, we start with a fresh log screen:

##### 4.1 Log Screen

In [ ]:
# LOG SCREEN (VLC Chat)

import sys, threading, html
import ipywidgets as widgets
from IPython.display import display
from collections import deque

BG = "#d6dde1"     # background (light blue)
FG = "#3e4f58"     # foreground (text) color
HEIGHT = 400       # px
TAIL_LINES = 400   # show last N lines

# 1) Render the log as HTML with inline styles (works in VS Code)
_STYLE = (
    f"background:{BG}; color:{FG}; height:{HEIGHT}px; overflow-y:auto; "
    "border:0; padding:8px; font-family:ui-monospace,monospace; "
    "font-size:10px; line-height:1.25;"
)
log = widgets.HTML(value=f'<div style="{_STYLE}"><pre style="margin:0; color:inherit;"></pre></div>')
display(log)

# 2) Tail buffer that updates the HTML (no print -> no recursion)
class _TailHTML:
    def __init__(self, widget, max_lines=TAIL_LINES):
        self.w = widget
        self.max_lines = max_lines
        self.lines = deque(maxlen=max_lines)
        self.partial = ""
        self.lock = threading.Lock()

    def _render(self):
        text = "\n".join(list(self.lines) + ([self.partial] if self.partial else []))
        esc = html.escape(text)
        self.w.value = f'<div style="{_STYLE}"><pre style="margin:0; color:inherit;">{esc}</pre></div>'

    def write(self, s):
        if not s: return
        with self.lock:
            if "\r" in s:
                s = s.split("\r")[-1]
                self.partial = ""
            self.partial += s
            while "\n" in self.partial:
                line, self.partial = self.partial.split("\n", 1)
                self.lines.append(line)
            self._render()

    def clear(self):
        with self.lock:
            self.lines.clear()
            self.partial = ""
            self._render()

TAIL = _TailHTML(log, max_lines=TAIL_LINES)

# 3) Hook stdout (optionally stderr) into the HTML log
_ORIG_STDOUT, _ORIG_STDERR = sys.stdout, sys.stderr

class _Sink:
    def __init__(self, tail): self.tail = tail
    def write(self, s): self.tail.write(s)
    def flush(self): pass

sys.stdout = _Sink(TAIL)
# To capture stderr too, uncomment:
# sys.stderr = _Sink(TAIL)

def stop_capture():
    sys.stdout = _ORIG_STDOUT
    sys.stderr = _ORIG_STDERR
    print("Capture stopped.")

def clear_log():
    TAIL.clear()

print( "VLC Chat:")

##### 4.2 Chat Console

In [ ]:
DEST_ADDR = "CD"            # <— set once
DEFAULT_PORT = "A"       # "A" -> sendA, "B" -> sendB

import html, threading
import ipywidgets as widgets
from IPython.display import display
from collections import deque

# Look & feel
BG, FG = "#d6dde1", "#3e4f58"
HEIGHT, MAX_LINES = 220, 400
_style = (
    f"background:{BG}; color:{FG}; height:{HEIGHT}px; overflow-y:auto; "
    "border:1; padding:8px; font-family:ui-monospace,monospace; "
    "font-size:10px; line-height:1.25;"
)

# Chat log (HTML so styles work in VS Code)
_log = widgets.HTML(value=f'<div style="{_style}"><pre style="margin:0;color:inherit;"></pre></div>')
display(_log)

# Tail buffer (autoscroll effect)
_lines, _lock = deque(maxlen=MAX_LINES), threading.Lock()
def _render(text):
    _log.value = f'<div style="{_style}"><pre style="margin:0;color:inherit;">{html.escape(text)}</pre></div>'
def _append(line):
    with _lock:
        _lines.append(line)
        _render("\n".join(_lines))

# Input & controls
msg = widgets.Text(placeholder="Type message…", layout=widgets.Layout(flex="1"))
send_btn = widgets.Button(description="Send", button_style="primary")
port_sel = widgets.ToggleButtons(options=[("Port A","A"),("Port B","B")], value=DEFAULT_PORT)
clear_btn = widgets.Button(description="Clear")
hdr = widgets.HBox([widgets.HTML(f"<b>To:</b> {DEST_ADDR}&nbsp;"), port_sel, clear_btn])
row = widgets.HBox([msg, send_btn])
display(hdr, row)

# Sending
def _send_text(t):
    cmd = f"m[{t}\0,{DEST_ADDR}]"
    try:
        if port_sel.value == "A":
            sendA(cmd)
        else:
            sendB(cmd)
    except NameError:
        _append("[sys] sendA/sendB not defined; simulated send.")

# Send handlers
_ignore_change = False  # prevents double-send on programmatic clears
def on_send(_=None):
    global _ignore_change
    t = msg.value.strip()
    if not t: return
    _append(f"[you → {DEST_ADDR}] {t}")
    _send_text(t)
    _ignore_change = True
    msg.value = ""     # triggers observe; we ignore it
    _ignore_change = False

send_btn.on_click(on_send)

# Enter-to-send without on_submit (ipywidgets 8)
# Only commit on Enter/blur:
if hasattr(msg, "continuous_update"):
    msg.continuous_update = False
def _commit_handler(change):
    if _ignore_change: return
    if change["name"] == "value" and change["type"] == "change":
        if msg.value.strip():
            on_send()
msg.observe(_commit_handler, names="value")

# Clear button
def on_clear(_):
    with _lock:
        _lines.clear()
        _render("")
clear_btn.on_click(on_clear)

# Optional: hook for your serial pump to display incoming lines here
def chat_rx(port_name, line):
    _append(f"[{port_name} RX] {line}")

_append("[sys] Chat ready. Type and press Enter or click Send.")


### 5. Performance evaluation of LED-to-LED communication: delay, coverage distance, saturation throughput

##### 5.1 Log Screen

Again, a new log screen to capture all the printed lines:

In [ ]:
# LOG SCREEN (VLC Performance)

import sys, threading, html
import ipywidgets as widgets
from IPython.display import display
from collections import deque

BG = "#d6dde1"     # background (light blue)
FG = "#3e4f58"     # foreground (text) color
HEIGHT = 400       # px
TAIL_LINES = 400   # show last N lines

# 1) Render the log as HTML with inline styles (works in VS Code)
_STYLE = (
    f"background:{BG}; color:{FG}; height:{HEIGHT}px; overflow-y:auto; "
    "border:0; padding:8px; font-family:ui-monospace,monospace; "
    "font-size:10px; line-height:1.25;"
)
log = widgets.HTML(value=f'<div style="{_STYLE}"><pre style="margin:0; color:inherit;"></pre></div>')
display(log)

# 2) Tail buffer that updates the HTML (no print -> no recursion)
class _TailHTML:
    def __init__(self, widget, max_lines=TAIL_LINES):
        self.w = widget
        self.max_lines = max_lines
        self.lines = deque(maxlen=max_lines)
        self.partial = ""
        self.lock = threading.Lock()

    def _render(self):
        text = "\n".join(list(self.lines) + ([self.partial] if self.partial else []))
        esc = html.escape(text)
        self.w.value = f'<div style="{_STYLE}"><pre style="margin:0; color:inherit;">{esc}</pre></div>'

    def write(self, s):
        if not s: return
        with self.lock:
            if "\r" in s:
                s = s.split("\r")[-1]
                self.partial = ""
            self.partial += s
            while "\n" in self.partial:
                line, self.partial = self.partial.split("\n", 1)
                self.lines.append(line)
            self._render()

    def clear(self):
        with self.lock:
            self.lines.clear()
            self.partial = ""
            self._render()

TAIL = _TailHTML(log, max_lines=TAIL_LINES)

# 3) Hook stdout (optionally stderr) into the HTML log
_ORIG_STDOUT, _ORIG_STDERR = sys.stdout, sys.stderr

class _Sink:
    def __init__(self, tail): self.tail = tail
    def write(self, s): self.tail.write(s)
    def flush(self): pass

sys.stdout = _Sink(TAIL)
# To capture stderr too, uncomment:
# sys.stderr = _Sink(TAIL)

def stop_capture():
    sys.stdout = _ORIG_STDOUT
    sys.stderr = _ORIG_STDERR
    print("Capture stopped.")

def clear_log():
    TAIL.clear()

print( "VLC Performance:")


Use the red LEDs in the following. Use one device as a transmitter and one as a receiver, ideally connected to two different machines or the same machine. Point the LEDs directly at each other to communicate over longer distances.

<center>
<img src="../../Admin/picture/vlc_measurement.png" 
     alt="VLC Measurement" 
     width="40%" 
     style="filter: grayscale(40%);">
     <p><em>Measurement setup. For this evaluation, we do not need precise inter-LED distances; an estimate like “around 10 cm” is sufficient.</em></p>
</center>

##### 5.2 Performance test

From one of the two devices, send packets manually: Send a new message a few times after you receive the message done command (`m[D]`) over serial. The code cell below does this for you, but we have to make sure the configuration parameters are correct, especially the destination address. Run the same experiment for a packet payload size of 1 byte and 180 byte (200 byte is the maximum possible payload) for answering the questions below. Log the transmission success (ACK received or not) and the distance a few times. Also log the round-trip delay (RTT) - the time from transmit to receipt of the corresponding `ACK`, measured at the transmitting device, and determine the maximum possible throughput (the saturation throughput). Look out for a logged line like this: `ACK seq=55  dispatch_ms=698614  rtt_ms=1759.000  (#3)`. The round trip time can be taken from `rtt_ms`.

In [ ]:
# VLC eval using sendA/sendB
import time, random, string, threading

# ---- Config ----
TX_PORT      = "A"       # "A" -> sendA, "B" -> sendB
DEST_ADDR    = "CD"      # target (e.g., node ID or 'CD')
PAYLOAD_SIZE = 1       # bytes (e.g., 1, 100, 180)
DURATION_S   = 10       # total run time
SEND_EVERY_S = 3       # send rate (s). Set None to only send on device prompts.

# ---- Require sendA/sendB ----
if "sendA" not in globals() or "sendB" not in globals():
    raise RuntimeError("sendA/sendB not found. Define them first.")

# ---- Payload (safe ASCII, no brackets/commas) ----
SAFE = string.ascii_letters + string.digits + " "
payload = "".join(random.choices(SAFE, k=PAYLOAD_SIZE)) if PAYLOAD_SIZE > 0 else ""

# ---- State ----
dispatch_times = {}   # seq -> dispatch time (ms)
row_count = 0
_stop = threading.Event()

def _send(text):
    msg = f"m[{text}\0,{DEST_ADDR}]\n"
    (sendA if TX_PORT == "A" else sendB)(msg)

def _sender_loop():
    if SEND_EVERY_S is None:
        return
    next_t = time.time()
    while not _stop.is_set():
        _send(payload)
        next_t += SEND_EVERY_S
        _stop.wait(max(0.0, next_t - time.time()))

# ---- Hook called by pump() for each complete incoming line ----
def on_serial_line(port_name, line):
    """
    Expects:
      s[T,D,...] -> contains dispatch time
      s[R,A,...] -> ACK with receive time
    Indexing: stats[4]=seq, stats[7]=tx_ms, stats[8]=rx_ms
    """
    global row_count
    try:
        if not line:
            return
        stats = (line[2:-1]).split(',') if len(line) > 3 else []
        head5 = line[:5] if len(line) >= 5 else ""

        if head5 == "m[D]":
            _send(payload)
            return

        if head5.startswith("s[T,D") and len(stats) >= 9:
            seq   = stats[4]
            tx_ms = stats[7]
            dispatch_times[seq] = tx_ms
            return

        if head5.startswith("s[R,A") and len(stats) >= 9:
            seq   = stats[4]
            rx_ms = stats[8]
            tx_ms = dispatch_times.pop(seq, None)
            if tx_ms is not None:
                rtt = float(rx_ms) - float(tx_ms)
                row_count += 1
                print(f"ACK seq={seq}  dispatch_ms={tx_ms}  rtt_ms={rtt:.3f}  (#{row_count})")
            return
    except Exception as e:
        print(f"[{port_name} parse warn] {e}")

# ---- Run ----
# Kick off
_send(payload)

# Periodic sender (if configured)
t = threading.Thread(target=_sender_loop, daemon=True)
t.start()

t0 = time.time()
try:
    while time.time() - t0 < DURATION_S:
        time.sleep(0.05)  # work happens via on_serial_line hook
except KeyboardInterrupt:
    pass
finally:
    _stop.set()
    t.join(timeout=1.0)

elapsed = max(1e-6, time.time() - t0)
print(f"Done. Packets logged: {row_count}  |  Duration: {elapsed:.1f}s  |  Rate: {row_count/elapsed:.2f} packets/s")